In [1]:
from datasets import load_from_disk
from pathlib import Path
project_path = Path.cwd().parent
dataset_path = project_path/"data"/"test_dataset"
dataset = load_from_disk(dataset_path)

d:\Anaconda\envs\diffusion\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import PeftModel
import os
import glob
base_mode = "Qwen/Qwen2.5-1.5B"
base_model = AutoModelForCausalLM.from_pretrained(base_mode, dtype = torch.bfloat16, device_map = "auto")
checkpoint_folder = glob.glob(os.path.join(project_path,"outputs/sft", "checkpoint-*"))
model = PeftModel.from_pretrained(base_model, checkpoint_folder[0], adapter_name="sft")

tokenizer = AutoTokenizer.from_pretrained(base_mode)

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 188.26it/s]
Exception in thread Thread-6 (_readerthread):
Traceback (most recent call last):
  File "d:\Anaconda\envs\diffusion\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "d:\Anaconda\envs\diffusion\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "d:\Anaconda\envs\diffusion\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
  File "d:\Anaconda\envs\diffusion\lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xb2 in position 7: invalid start byte


In [3]:
# set the padding_size, default is right, but in inference, we need left padding
tokenizer.pad_token
model.config.pad_token_id = tokenizer.pad_token_id
tokenizer.padding_side = 'left'

In [4]:
def predict(model, tokenizer, batch):
    input =  tokenizer(batch["prompt"], return_tensors = "pt", padding = True,  truncation = True).to("cuda")
    model.eval()
    with torch.no_grad():
        output = model.generate(
            **input, 
            max_new_tokens = 256, 
            do_sample = True,
            top_p = 0.7,
            temperature = 1.1, 
            repetition_penalty = 1.1
            )
    input_len = input.input_ids.shape[1]
    responses = [
        tokenizer.decode(reponse[input_len:], skip_special_tokens = True)
        for reponse in output
    ]
    return {"response" : responses}



In [ ]:
import pandas as pd
import random

def judge(sft_res_dataset, dpo_res_dataset, beta_val): 
    df_sft = sft_res_dataset.to_pandas()
    df_dpo = dpo_res_dataset.to_pandas()
    eval_data = []
    gpt_juge_data = []

    for i in range(len(df_sft)):
        prompt = df_sft.iloc[i]["prompt"]
        ans_sft = df_sft.iloc[i]["response"]
        ans_dpo = df_dpo.iloc[i]["response"]

        if random.random() > 0.5 :
            choice_a, choice_b = ans_sft, ans_dpo
            winner_is_a = "SFT"
        else :
            choice_a, choice_b = ans_dpo, ans_sft
            winner_is_a = "DPO"
        eval_data.append({
            "instruction" : prompt,
            "answer_a" : choice_a,
            "answer_b" : choice_b,
            "truth" : winner_is_a
        })
        gpt_juge_data.append({
            "instruction" : prompt,
            "answer_a" : choice_a,
        "answer_b" : choice_b,
    })

    eval_df = pd.DataFrame(eval_data)
    eval_df.to_json(project_path/"outputs"/"eval"/f"results_beta_{str(beta_val).replace('.','_')}.jsonl", orient = "records", lines = True, force_ascii = False)
    gpt_df = pd.DataFrame(gpt_juge_data)
    gpt_df.to_json(project_path/"outputs"/"eval"/f"gpt_judge_beta_{str(beta_val).replace('.','_')}.jsonl", orient = "records", lines = True, force_ascii = False)

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=""
)

## here, because the dataset is in Chinese, so we use Chinses system prompt to ask to GPT
def get_judge_result(instruction, ans_a, ans_b):
    """调用 gpt-4o-mini 进行打分"""
    sys_prompt = "你是一位专业的 LLM 评测专家。请对比两个 AI 对同一个问题的回答，并选出更好的一个。"
    user_prompt = f"""【问题】: {instruction}

【回答 A】: {ans_a}
【回答 B】: {ans_b}

请从有用性、逻辑性、指令遵循度进行评价。最后一行请务必给出结论：
- 如果 A 更好，输出 [[A]]
- 如果 B 更好，输出 [[B]]
- 如果平局，输出 [[Tie]]"""

    try:
        response = client.chat.completions.create(
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": user_prompt},
            ],
            model="Phi-4",
            temperature=0  # 裁判不需要创造力，只要稳定性
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"请求出错了: {e}")
        return "Error"

In [ ]:
import json
import time
import pandas as pd
def gpt_judge(beta_val):
    with open(project_path/"outputs"/"eval"/f"results_beta_{str(beta_val).replace('.','_')}.jsonl", "r", encoding="utf-8") as f:
        eval_list = [json.loads(line) for line in f]


    final_results = []
    print(f"🚀 Starting Evaluating Number : {len(eval_list)} data...")
    file_path = project_path/"outputs"/"eval"/f"final_evaluation_results_beta_{str(beta_val).replace('.','_')}.jsonl"

    processed_prompts = set()
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    data = json.loads(line)
                    processed_prompts.add(data['prompt'])
                except json.JSONDecodeError:
                    continue
        print(f"Existing progress detected： Skip {len(processed_prompts)} data")

    for i, item in enumerate(eval_list):

        if item["instruction"] in processed_prompts :
            continue

        print(f"in evaluating the  {i+1}/45 data...")
        
        feedback = get_judge_result(item['instruction'], item['answer_a'], item['answer_b'])
        
        if "[[A]]" in feedback:
            choice = "A"
        elif "[[B]]" in feedback:
            choice = "B"
        else:
            choice = "Tie"
        
        truth_label = item['truth'] # turth represent the A is from DPO or SFT
        if choice == "Tie":
            winner = "Tie"
        elif choice == "A":
            winner = truth_label
        else :
            winner = "SFT" if item["truth"] == "DPO" else "DPO"

        final_results.append({
            "prompt": item['instruction'],
            "winner": winner,
            "reason": feedback
        })
        current_res = {
            "prompt": item['instruction'],
            "winner": winner,
            "reason": feedback
        }
        with open(file_path, "a", encoding = "utf-8") as f:
            f.write(json.dumps(current_res, ensure_ascii=False) + "\n")
        
        # we use github gpt api
        time.sleep(3) 
    # 4. win rate for different tasks
    final_results = pd.read_json(file_path, lines=True)
    task_ind = list(range(0,len(final_results), 15))
    task_ind.append(len(final_results))
    stats = pd.DataFrame(final_results[task_ind[0]:task_ind[1]])['winner'].value_counts()
    # You can compare the win rates for different types of tasks
    
    # print("\n" + "="*20)
    # print("📊 最终实验结果 (格式转换):")
    # print(stats)
    # print("="*20)
    # stats = pd.DataFrame(final_results[task_ind[1]:task_ind[2]])['winner'].value_counts()
    # print("\n" + "="*20)
    # print("📊 最终实验结果 (风格要求):")
    # print(stats)
    # print("="*20)

    # stats = pd.DataFrame(final_results[task_ind[2]:task_ind[3]])['winner'].value_counts()
    # print("\n" + "="*20)
    # print("📊 最终实验结果 (指示要求):")
    # print(stats)
    # print("="*20)
    stats = pd.DataFrame(final_results)['winner'].value_counts()
    print("\n" + "="*20)
    print("📊 最终实验结果 (指示要求):")
    print(stats)
    print("="*20)




In [ ]:
## results of base
with model.disable_adapter():
    base_res_dataset = dataset["test"].map(lambda x : predict(model, tokenizer, x), batched = True, batch_size = 8)

model.set_adapter("sft")
sft_res_dataset = dataset["test"].map(lambda x : predict(model, tokenizer, x), batched = True, batch_size = 8)
beta_list = [0.3,0.4,0.5] ## we test the three beta in the DPO training

    

Map: 100%|██████████| 45/45 [00:17<00:00,  2.53 examples/s]


In [9]:
import gc
beta_list = [0.5]
for beta_val in beta_list:
    model = PeftModel.from_pretrained(base_model, checkpoint_folder[0], adapter_name="sft")
    adapter_name = f"dpo_beta_{str(beta_val).replace('.', '_')}"
    output_dir = os.path.join(project_path, "outputs/dpo", f"dpo_beta_{str(beta_val).replace('.', '_')}")
    checkpoint_folder = glob.glob(os.path.join(output_dir, "checkpoint-*"))
    model.load_adapter(checkpoint_folder[0], adapter_name=adapter_name)
    model.set_adapter(f"dpo_beta_{str(beta_val).replace('.', '_')}")
    dpo_res_dataset = dataset["test"].map(lambda x : predict(model, tokenizer, x), batched = True, batch_size = 8)
    judge(sft_res_dataset, dpo_res_dataset, beta_val)
    gpt_judge(beta_val)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    

d:\Anaconda\envs\diffusion\lib\site-packages\peft\tuners\tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
Map: 100%|██████████| 45/45 [00:12<00:00,  3.48 examples/s]


🚀 Starting Evaluating Number : 45 data...
检测到已有进度：已跳过 10 条数据。
in evaluating the  1/45 data...
in evaluating the  2/45 data...
in evaluating the  3/45 data...
in evaluating the  4/45 data...
in evaluating the  5/45 data...
in evaluating the  6/45 data...
in evaluating the  7/45 data...
in evaluating the  8/45 data...
in evaluating the  9/45 data...
in evaluating the  10/45 data...
in evaluating the  11/45 data...
in evaluating the  12/45 data...
in evaluating the  13/45 data...
in evaluating the  14/45 data...
in evaluating the  15/45 data...
in evaluating the  16/45 data...
in evaluating the  17/45 data...
in evaluating the  18/45 data...
in evaluating the  19/45 data...
in evaluating the  20/45 data...
in evaluating the  21/45 data...
in evaluating the  22/45 data...
in evaluating the  23/45 data...
in evaluating the  24/45 data...
in evaluating the  25/45 data...
in evaluating the  26/45 data...
in evaluating the  27/45 data...
in evaluating the  28/45 data...
in evaluating the  29/4